In [1]:
from datasets import load_dataset

/kaggle/text2phoneme/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
interleave = load_dataset("/kaggle/text2phoneme/interleaved_vi_en/interleaved_subset", split="train")
print(interleave[0]['text'])

Là một <en>office worker</en>, Nguyễn Thị Hoa có <en>experience in using basic software</en> và <en>processing documents</en>. Bà là người <en>reliable</en>, có <en>ability to collaborate with colleagues</en> và đảm bảo <en>tasks are completed on time</en>.


In [6]:
vietnamese = load_dataset("/kaggle/text2phoneme/interleaved_vi_en/vietnamese_subset", split="train")
print(vietnamese[0]['text'])

Generating train split: 863000 examples [00:00, 2397825.17 examples/s]

Generating train split: 1000000 examples [00:00, 2310529.39 examples/s]

Là một nhân viên văn phòng, Nguyễn Thị Hoa có kinh nghiệm trong việc sử dụng các phần mềm cơ bản và xử lý tài liệu. Bà là người đáng tin cậy, có khả năng hợp tác với đồng nghiệp và đảm bảo công việc được hoàn thành đúng hạn.


In [8]:
english = load_dataset("/kaggle/text2phoneme/interleaved_vi_en/english_subset", split="train")
print(english[0]['text'])

Generating train split: 431000 examples [00:00, 1681755.11 examples/s]

Generating train split: 10003212 examples [00:06, 1501599.79 examples/s]


Mary Alberti is a front‑line food service specialist whose razor‑sharp cash handling, inventory tracking, and POS mastery combine with a disciplined, routine‑driven work ethic, enabling them to calmly resolve high‑pressure customer issues and hit performance targets while eyeing a promotion to shift supervisor.


In [ ]:
import logging
from abc import ABC, abstractmethod
from typing import List, Optional
from functools import reduce
from typing import Dict
import re
try:
    from piper_phonemize import phonemize_espeak
except Exception as ex:
    raise RuntimeError(
        f"{ex}\nPlease run\n"
        "pip install piper_phonemize -f \
            https://k2-fsa.github.io/icefall/piper_phonemize.html"
    )


class Tokenizer(ABC):
    """Abstract base class for tokenizers, defining common interface."""

    @abstractmethod
    def texts_to_token_ids(self, texts: List[str]) -> List[List[int]]:
        """Convert list of texts to list of token id sequences."""
        raise NotImplementedError

    @abstractmethod
    def texts_to_tokens(self, texts: List[str]) -> List[List[str]]:
        """Convert list of texts to list of token sequences."""
        raise NotImplementedError

    @abstractmethod
    def tokens_to_token_ids(self, tokens: List[List[str]]) -> List[List[int]]:
        """Convert list of token sequences to list of token id sequences."""
        raise NotImplementedError


class EspeakTokenizer(Tokenizer):
    """A simple tokenizer with Espeak g2p function."""

    def __init__(self, token_file: Optional[str] = None, lang: str = "en-us"):
        """
        Args:
          tokens: the file that contains information that maps tokens to ids,
            which is a text file with '{token}\t{token_id}' per line.
          lang: the language identifier, see
            https://github.com/rhasspy/espeak-ng/blob/master/docs/languages.md
        """
        # Parse token file
        self.has_tokens = False
        self.lang = lang
        if token_file is None:
            return
        self.token2id: Dict[str, int] = {}
        with open(token_file, "r", encoding="utf-8") as f:
            for line in f.readlines():
                info = line.rstrip().split("\t")
                token, id = info[0], int(info[1])
                assert token not in self.token2id, token
                self.token2id[token] = id
        self.pad_id = self.token2id["_"]  # padding
        self.vocab_size = len(self.token2id)
        self.has_tokens = True

    def g2p(self, text: str) -> List[str]:
        try:
            tokens = phonemize_espeak(text, self.lang)
            tokens = reduce(lambda x, y: x + y, tokens)
            return tokens
        except Exception as ex:
            logging.warning(f"Tokenization of {self.lang} texts failed: {ex}")
            return []

    def texts_to_token_ids(
        self,
        texts: List[str],
    ) -> List[List[int]]:
        return self.tokens_to_token_ids(self.texts_to_tokens(texts))

    def texts_to_tokens(
        self,
        texts: List[str],
    ) -> List[List[str]]:
        tokens_list = [self.g2p(texts[i]) for i in range(len(texts))]
        return tokens_list

    def tokens_to_token_ids(
        self,
        tokens_list: List[List[str]],
    ) -> List[List[int]]:
        assert self.has_tokens, "Please initialize Tokenizer with a tokens file."

        token_ids_list = []

        for tokens in tokens_list:
            token_ids = []
            for t in tokens:
                if t not in self.token2id:
                    logging.debug(f"Skip OOV {t}")
                    continue
                token_ids.append(self.token2id[t])

            token_ids_list.append(token_ids)

        return token_ids_list

vi_tok = EspeakTokenizer(lang="vi", token_file="/kaggle/text2phoneme/resource/phoneme_vi.txt")
en_tok = EspeakTokenizer(lang="en", token_file="/kaggle/text2phoneme/resource/phoneme_en.txt")


In [15]:
en_tok.texts_to_token_ids([english[0]['text']])

[[25,
  120,
  18,
  59,
  88,
  21,
  3,
  14,
  24,
  15,
  120,
  62,
  122,
  32,
  21,
  3,
  74,
  38,
  3,
  50,
  3,
  19,
  88,
  120,
  102,
  26,
  32,
  3,
  24,
  120,
  14,
  74,
  26,
  3,
  19,
  120,
  33,
  122,
  17,
  3,
  31,
  120,
  62,
  122,
  34,
  74,
  31,
  3,
  31,
  28,
  120,
  61,
  96,
  59,
  24,
  121,
  74,
  31,
  32,
  3,
  20,
  121,
  33,
  122,
  38,
  3,
  88,
  120,
  18,
  74,
  38,
  59,
  3,
  96,
  120,
  51,
  122,
  28,
  3,
  23,
  120,
  14,
  96,
  3,
  20,
  120,
  14,
  26,
  17,
  24,
  74,
  44,
  8,
  3,
  120,
  74,
  26,
  34,
  59,
  26,
  32,
  88,
  21,
  3,
  32,
  88,
  120,
  14,
  23,
  74,
  44,
  8,
  3,
  14,
  26,
  17,
  3,
  28,
  121,
  21,
  122,
  121,
  59,
  100,
  120,
  61,
  31,
  3,
  25,
  120,
  14,
  31,
  32,
  59,
  88,
  21,
  3,
  23,
  59,
  25,
  15,
  120,
  14,
  74,
  26,
  3,
  35,
  74,
  41,
  3,
  50,
  3,
  17,
  120,
  74,
  31,
  74,
  28,
  24,
  121,
  74,
  26,
  17,
  8,
  3,
  88,


In [16]:
vi_tok.texts_to_token_ids([vietnamese[0]['text']])

[[24,
  121,
  14,
  122,
  132,
  3,
  25,
  120,
  27,
  136,
  32,
  142,
  3,
  82,
  120,
  59,
  26,
  3,
  34,
  120,
  21,
  61,
  26,
  3,
  34,
  120,
  14,
  26,
  3,
  19,
  120,
  54,
  132,
  44,
  8,
  3,
  44,
  35,
  120,
  21,
  61,
  135,
  26,
  3,
  32,
  120,
  21,
  136,
  3,
  20,
  35,
  120,
  14,
  122,
  3,
  23,
  120,
  54,
  62,
  3,
  23,
  120,
  21,
  82,
  3,
  44,
  120,
  21,
  61,
  136,
  25,
  3,
  32,
  96,
  120,
  54,
  44,
  3,
  34,
  120,
  21,
  61,
  136,
  16,
  3,
  31,
  120,
  37,
  134,
  3,
  38,
  120,
  33,
  136,
  44,
  3,
  23,
  121,
  14,
  122,
  62,
  16,
  3,
  19,
  120,
  59,
  132,
  26,
  3,
  25,
  120,
  18,
  132,
  25,
  3,
  23,
  120,
  59,
  122,
  3,
  15,
  120,
  14,
  122,
  134,
  26,
  3,
  34,
  121,
  14,
  122,
  132,
  3,
  31,
  120,
  37,
  134,
  3,
  24,
  120,
  21,
  62,
  3,
  32,
  142,
  120,
  14,
  122,
  132,
  22,
  3,
  24,
  120,
  21,
  61,
  136,
  35,
  10,
  15,
  120,
  14,
  122,
 